<a href="https://colab.research.google.com/github/Shrutakeerti/Sentiment-Analysis-Info/blob/main/Autoencoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from keras.models import Model
from keras.layers import Input, Dense

nltk.download('stopwords')
from nltk.corpus import stopwords

# Step 1: Data Collection and Preprocessing
# Dummy dataset
data = {
    'post': [
        'I love this product!',
        'This is the worst service ever.',
        'Amazing experience, will come again.',
        'I am very disappointed.',
        'Totally worth it!',
        'Never coming back here.'
    ],
    'reply': [
        'Me too, it’s fantastic!',
        'I agree, it’s terrible.',
        'Absolutely wonderful!',
        'Same here, very let down.',
        'Indeed, it’s great!',
        'Terrible experience.'
    ]
}

df = pd.DataFrame(data)

# Preprocessing function
def preprocess_text(text):
    text = re.sub(r'\W', ' ', str(text))
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = text.strip()
    return text

# Apply preprocessing
df['post'] = df['post'].apply(preprocess_text)
df['reply'] = df['reply'].apply(preprocess_text)

# Combine post and reply for analysis
df['post_reply'] = df['post'] + ' ' + df['reply']

# Step 2: Feature Extraction
vectorizer = TfidfVectorizer(stop_words=stopwords.words('english'), max_features=5000)
X = vectorizer.fit_transform(df['post_reply']).toarray()

# Step 3: Autoencoder Model
input_dim = X.shape[1]
encoding_dim = 128

input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation='relu')(input_layer)
decoder = Dense(input_dim, activation='sigmoid')(encoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train the autoencoder
autoencoder.fit(X, X, epochs=50, batch_size=32, shuffle=True, validation_split=0.2)

# Extract the encoder part for dimensionality reduction
encoder_model = Model(inputs=input_layer, outputs=encoder)
encoded_X = encoder_model.predict(X)

# Step 4: Clustering
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(encoded_X)

# Assign clusters to the original data
df['cluster'] = clusters

# Step 5: Evaluation
silhouette_avg = silhouette_score(encoded_X, clusters)
print(f'Silhouette Score: {silhouette_avg}')

# Display the clustered data
print(df)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Epoch 1/50
1/1 [==============================] - 2s 2s/step - loss: 0.6908 - val_loss: 0.6883
Epoch 2/50
1/1 [==============================] - 0s 115ms/step - loss: 0.6859 - val_loss: 0.6853
Epoch 3/50
1/1 [==============================] - 0s 169ms/step - loss: 0.6809 - val_loss: 0.6823
Epoch 4/50
1/1 [==============================] - 0s 167ms/step - loss: 0.6761 - val_loss: 0.6793
Epoch 5/50
1/1 [==============================] - 0s 120ms/step - loss: 0.6712 - val_loss: 0.6763
Epoch 6/50
1/1 [==============================] - 0s 157ms/step - loss: 0.6664 - val_loss: 0.6733
Epoch 7/50
1/1 [==============================] - 0s 130ms/step - loss: 0.6616 - val_loss: 0.6704
Epoch 8/50
1/1 [==============================] - 0s 147ms/step - loss: 0.6568 - val_loss: 0.6675
Epoch 9/50
1/1 [==============================] - 0s 117ms/step - loss: 0.6520 - val_loss: 0.6646
Epoch 10/50
1/1 [==============================] - 0s 94ms/step - loss: 0.6473 - val_loss: 0.6617
Epoch 11/50
1/1 [======

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
